# 第 4 课 · 跑完并读懂一轮训练

**本课用时约 35 分钟。** 逐批执行采样、校验、策略更新，并按指令解释一轮结果。

## 学完后你能做到

- 查看一批采样前后的权重差。
- 区分尝试数、正确样本数与开发集准确率。
- 识别一轮训练里没有收集到正样本的组。

## 跟着操作

先运行小批次，再将预算从 16 改成 64，比较随机波动。

运行代码格时，把光标放在格子里，按 **Shift + Enter**。看到输出后再运行下一格。若第一次误点或修改了上方变量，选 Jupyter 菜单里的 **Kernel → Restart Kernel and Run All Cells**，从第一格重新运行。本课不要求预先懂强化学习，也不需要 GPU。

## 一次更新发生在哪里

本课将一批经历记录成 `(指令, 起点, 选择的规则, 模型答案)`。先用旧策略完整采样，然后统一检查这批轨迹。每个通过校验的轨迹给对应规则加一。更新是在一整批结束后发生，因此本批后半段不会使用刚刚更新的权重。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from rsi_lab import INITIAL, TRAIN, evaluate, sample_batch, update
import random

rng = random.Random(19)
batch = sample_batch(rng, INITIAL, [1, 1, 1, 1], budget=16)
# 此时 16 个答案都已生成，下面的更新只会影响下一批。
updated, attempts, kept = update(INITIAL, batch)
assert sum(attempts) == len(batch)
assert [sum(new) - sum(old) for new, old in zip(updated, INITIAL)] == kept
print("首条轨迹 (指令, 起点, 规则, 答案):", batch[0])
print("尝试:", attempts)
print("通过:", kept)
print("权重变化:")
for name, before, after in zip(("前进一步", "前进两格", "前进三格", "后退一步"), INITIAL, updated):
    print(name, before, "->", after)

尝试数回答「这类题出现了几次」，通过数回答「有几条正反馈」，权重差回答「策略被哪些正样本改变」。某一组的通过数为零时，这轮就没有正样本能教会它。后续轮次能否恢复取决于它是否还会被抽到、能否探索到正确候选。

In [ ]:
before = evaluate(INITIAL, range(50, 75))
after = evaluate(updated, range(50, 75))
print(f"开发集期望正确率: {before['expected']:.1%} -> {after['expected']:.1%}")
for name, old, new in zip(("前进一步", "前进两格", "前进三格", "后退一步"),
                          before["by_task"], after["by_task"]):
    print(f"{name}: {old:.1%} -> {new:.1%}")

期望正确率按各候选被抽中的概率计分。例如正确规则占 2/10，这道题就有 20% 的答对机会。贪心正确率只看权重最大的规则，遇到并列取第一条。正确规则权重增长但仍未排第一时，两种指标会有怎样的变化？

同一预算重复运行但换随机种子，可以看到有限样本下结果不同。记录种子、预算和本批通过数，才能解释结果差异。不要只选得分最高的一次。

In [ ]:
for seed in (19, 20, 21):
    local_rng = random.Random(seed)
    local_batch = sample_batch(local_rng, INITIAL, [1, 1, 1, 1], budget=16)
    local_weights, local_attempts, local_kept = update(INITIAL, local_batch)
    print(seed, "通过", local_kept, "开发期望", round(evaluate(local_weights, range(50, 75))["expected"], 3))

## 自检

- [ ] 更新后的权重只因通过校验的样本改变。
- [ ] 预算和尝试数不是正确样本数。
- [ ] 评估比较使用固定开发集。
- [ ] 我能为一个没有得到正样本的组提出失败原因。

下一课把训练分成多轮，看策略更新和练习分配如何形成递归反馈。

[返回课程首页并选择下一节](recursive_self_improvement.ipynb)